# PyHPCC SQL driver

A small example of the read-only WsSQL2 DB-API driver. Run All uses localhost over HTTP with user jypter_notebook, without prompting. Edit the configuration below to use another server. If a password is required, set HPCC_PASSWORD in the kernel environment rather than storing it in this file.

In [8]:
import os
from pathlib import Path
import sys

# Allow running directly from a source checkout with a notebook kernel that
# has not installed PyHPCC as a package.
for directory in (Path.cwd(), *Path.cwd().parents):
    source_dir = directory / "src"
    if (source_dir / "pyhpcc").is_dir():
        sys.path.insert(0, str(source_dir))
        break

from pyhpcc.models.auth import Auth
from pyhpcc.sql import connect

host = "localhost"
protocol = "http"
sql_port = 8510
username = "jypter_notebook"
password = os.environ.get("HPCC_PASSWORD", "")

# Auth.port is separate from the WsSQL2 service port configured below.
auth = Auth(
    host,
    "8010",
    username,
    password,
    require_auth=bool(username),
    protocol=protocol,
)
connection = connect(
    auth,
    sql_url=f"{protocol}://{host}:{sql_port}/WsSQL2",
    connect_timeout=10,
    read_timeout=60,
)

## A simple query

`SELECT 1` is a quick way to verify that the WsSQL2 connection works.

In [9]:
with connection.cursor() as cursor:
    cursor.execute("SELECT 1 AS value")
    print("Columns:", [column[0] for column in cursor.description])
    print("Row:", cursor.fetchone())

Columns: ['value']
Row: (1,)


## List logical files and inspect columns

The metadata helpers use WsDfu and return DB-API cursors. Adjust the pattern to match the logical files available on your cluster.

In [15]:
with connection.get_tables(table_pattern="regress::multi::%") as tables:
    table_rows = tables.fetchmany(10)

for row in table_rows:
    print(row[2], "-", row[3])

regress::multi::book - TABLE
regress::multi::bookauthorgrouped - TABLE
regress::multi::bookindex - TABLE
regress::multi::bookindexid - TABLE
regress::multi::c.dg_fetchfile - TABLE
regress::multi::c.dg_fetchfilepreload - TABLE
regress::multi::c.dg_fetchfilepreloadindexed - TABLE
regress::multi::dg_accountxml - TABLE
regress::multi::dg_child.d00 - TABLE
regress::multi::dg_csv - TABLE


In [11]:
logical_file = "regress::multi::book"  # Change to a logical file on your cluster.
with connection.get_columns(table_pattern=logical_file) as columns:
    for row in columns.fetchmany(20):
        print(f"{row[3]}: {row[5]}")

id: BIGINT
name: VARCHAR
author: VARCHAR
rating100: BIGINT
price: DECIMAL(8,2)
personid: BIGINT


## Read a few rows

This example uses the sample `regress::local::people` file. Change the logical file and selected columns as needed. The row limit prevents this example from fetching an unbounded result.

In [14]:
with connection.cursor() as cursor:
    cursor.set_max_rows(10)
    cursor.execute(
        "SELECT id, name, author FROM hpcc_scan('regress::multi::book')"
    )
    for row in cursor:
        print(row)

(1, 'To kill a mocking bird', 'Harper Lee')
(2, 'Clarion Language Manual', 'Richard Taylor')
(3, 'The bible', 'Various')
(4, 'Compilers', 'Aho, Sethi, Ullman')
(5, 'Lord of the Rings', 'JRR Tolkien')
(6, 'The thinks you can think', 'Dr. Seuss')
(7, 'Where is flop?', '')
(8, 'David and Goliath', '')
(9, 'The story of Jonah', '')
(10, 'The Life of Pi', 'Yan Martel')


Close the connection when finished.

In [6]:
connection.close()